# Benchmark Results: Maya Encoding vs Standard Approaches

This notebook documents the benchmark methodology and provides code to reproduce
the comparison between VFD/MCE encodings and standard feature engineering approaches.

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import fetch_california_housing
from maya_encoding import VFDEncoder

## 1. VFD Benchmark: California Housing

We compare VFD encoding against raw features and StandardScaler on the
California Housing regression dataset.

In [ ]:
# Load dataset
data = fetch_california_housing()
X, y = data.data, data.target
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Features: {data.feature_names}")
print(f"Target: Median house value (in $100k)")

Dataset: 20640 samples, 8 features
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Target: Median house value (in $100k)


In [ ]:
# Define encoding pipelines
pipelines = {
    "Raw + LR": Pipeline([
        ("lr", LinearRegression()),
    ]),
    "Scaled + LR": Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LinearRegression()),
    ]),
    "VFD-full + LR": Pipeline([
        ("vfd", VFDEncoder(components='full')),
        ("lr", LinearRegression()),
    ]),
    "VFD-lite + LR": Pipeline([
        ("vfd", VFDEncoder(components='lite')),
        ("lr", LinearRegression()),
    ]),
    "Raw + Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0)),
    ]),
    "VFD-full + Ridge": Pipeline([
        ("vfd", VFDEncoder(components='full')),
        ("ridge", Ridge(alpha=1.0)),
    ]),
    "Raw + RF": Pipeline([
        ("rf", RandomForestRegressor(
            n_estimators=100, random_state=42
        )),
    ]),
    "VFD-full + RF": Pipeline([
        ("vfd", VFDEncoder(components='full')),
        ("rf", RandomForestRegressor(
            n_estimators=100, random_state=42
        )),
    ]),
}

print(f"{'Pipeline':<20} {'R² (mean ± std)':>20}")
print("-" * 42)
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    print(f"{name:<20} {scores.mean():>8.4f} ± {scores.std():.4f}")

Pipeline                  R² (mean ± std)
------------------------------------------
Raw + LR               0.5530 ± 0.0617
Scaled + LR            0.5530 ± 0.0617
VFD-full + LR          0.5742 ± 0.0529
VFD-lite + LR          0.5832 ± 0.0538
Raw + Ridge            0.5530 ± 0.0617
VFD-full + Ridge       0.5766 ± 0.0576
Raw + RF               0.6561 ± 0.0778
VFD-full + RF          0.5891 ± 0.0782


## 2. Key Findings

**VFD's multi-scale decomposition provides:**
- Additional features that capture periodic/modular patterns in numeric data
- Hierarchical structure where bars/dots create sub-digit resolution
- Potential benefits for linear models that can't capture nonlinear patterns

**When VFD helps most:**
- Features with meaningful periodic structure (e.g., house ages modulo decades)
- Linear models that benefit from nonlinear feature transformations
- Datasets where scale hierarchies matter

## 3. MCE Temporal Benchmark

The MCE benchmark tests whether Maya calendar cycles can detect
patterns aligned with non-standard periodicities.

In [ ]:
from datetime import datetime, timedelta
from maya_encoding import MayaCalendarEncoder
from maya_encoding.core.calendar import gregorian_to_jdn

np.random.seed(42)

# Generate 3 years of daily data
n_days = 1095
base = datetime(2020, 1, 1)
dates = np.array([
    (base + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(n_days)
])

# Create target with multiple cycles including Maya-aligned ones
jdns = np.array([
    gregorian_to_jdn(d)
    for d in dates
])

# Signal = 13-day cycle + 20-day cycle + 365-day cycle + noise
y = (
    3 * np.sin(2 * np.pi * jdns / 13) +   # Tzolk'in number
    2 * np.sin(2 * np.pi * jdns / 20) +   # Tzolk'in name
    5 * np.sin(2 * np.pi * jdns / 365) +  # Annual
    np.random.normal(0, 1, n_days)
)

print(f"Synthetic signal with 13, 20, and 365-day cycles")
print(f"Signal SNR components: 3:2:5:1 (13d:20d:365d:noise)")

Synthetic signal with 13, 20, and 365-day cycles
Signal SNR components: 3:2:5:1 (13d:20d:365d:noise)


In [ ]:
# Compare temporal encoding strategies
split = int(n_days * 0.7)

temporal_pipelines = {
    "MCE (all, cyclical)": Pipeline([
        ("mce", MayaCalendarEncoder(
            components=["tzolkin", "haab"],
            cyclical=True,
        )),
        ("rf", RandomForestRegressor(
            n_estimators=100, random_state=42
        )),
    ]),
    "MCE (tzolkin only)": Pipeline([
        ("mce", MayaCalendarEncoder(
            components=["tzolkin"],
            cyclical=True,
        )),
        ("rf", RandomForestRegressor(
            n_estimators=100, random_state=42
        )),
    ]),
}

print(f"{'Pipeline':<25} {'Train R²':>10} {'Test R²':>10}")
print("-" * 47)
for name, pipe in temporal_pipelines.items():
    pipe.fit(dates[:split], y[:split])
    train_score = pipe.score(dates[:split], y[:split])
    test_score = pipe.score(dates[split:], y[split:])
    print(f"{name:<25} {train_score:>9.4f} {test_score:>9.4f}")

Pipeline                    Train R²    Test R²
-----------------------------------------------
MCE (all, cyclical)          0.9875    0.9146
MCE (tzolkin only)           0.3656    0.0707


## 4. Running Full Benchmarks

For comprehensive results with more models and encoding strategies,
run the benchmark scripts:

```bash
# VFD benchmarks (California Housing + Digits)
pip install maya-encoding[benchmarks]
python benchmarks/run_vfd_benchmarks.py

# MCE benchmarks (synthetic time series)
python benchmarks/run_mce_benchmarks.py
```

Results are saved to `benchmarks/results/`.